## first let's setup a agent with Nvidia's NIM

In [32]:
import os
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [33]:
# Setting up the NVIDIA NIM client
client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

In [34]:
agent = Agent(
    name="Assistant",
    instructions=(
        "You are a helpful AI assistant. "
        "Your goal is to provide accurate, clear, and concise answers. "
        "Break down complex topics into simple explanations. "
        "Ask for clarification if the question is unclear. "
        "Do not make up information—if unsure, say 'I don’t know.'"
    ),
    model=OpenAIChatCompletionsModel(
        model="google/gemma-3-1b-it",  
        openai_client=client
    ),
)

In [39]:
result = await Runner.run( 
    starting_agent=agent, # passing the agent to starting_agent
    input="What is the capital of France explain more about it?"
)

In [40]:
result.final_output

'Okay, let’s talk about the capital of France!\n\n**What is the capital of France?**\n\nThe capital of France is **Paris**.\n\n**Let’s break it down:**\n\n*   **What is Paris?** Paris is the largest city in France and one of the most famous cities in the world. It’s a truly global center for art, fashion, gastronomy, culture, and politics. \n\n*   **Why is it the capital?**  Historically, Paris has been the political and administrative center of France for centuries. It’s where the French government is located, and it’s where most major decisions are made. \n\n*   **Key Features:**\n    *   **Iconic Landmarks:**  The Eiffel Tower, Notre-Dame Cathedral, Louvre Museum, Arc de Triomphe, and numerous other stunning buildings are all located here.\n    *   **Culture:** Paris is renowned for its museums (the Louvre is the most famous), theaters, music venues, and a vibrant arts scene.\n    *   **Gastronomy:**  French cuisine is world-famous, and Paris is the perfect place to experience it – 

OPENAI_API_KEY is not set, skipping trace export


# Let's start working on the Multi-Agents

We will build a multi-agent system structured with a orchestrator-subagent pattern. The **orchestrator** in such a system refers to an agent that controls which _subagents_ are used and in which order, this orchestrator also handles all in / out communication with the users of a the system. The **subagent** is an agent that is built to handle a particular scenario or task. The subagent is triggered by the orchestrator and responds to the orchestrator when it is finished.

<img src="https://github.com/aurelio-labs/agents-sdk-course/blob/main/assets/handoffs-orchestrator-subagents-dark.png?raw=1" alt="Orchestrator-subagent pattern" width="1000"/>

# Sub Agents
We'll begin by defining our subagents. We will create three subagents, those are:

- **Web Search Subagent** will have access to the Linkup web search tool.

- **Internal Docs Subagent** will have access to some "internal" company documents.

- **Code Execution Subagent** will be able to write and execute simple Python code for us.

#### 1. Web Search Agent using Linkup

The web search subagent will take a user's query and use it to search the web. The agent will collect information from various sources and then merge that into a single text response that will be passed back to our orchestrator.

OpenAI's built-in web search is not great, so we'll use another web search API called [LinkUp](https://app.linkup.so/?utm_source=james). This service does require an account, but you will receive more than enough free credits to follow the course.

We initialize our Linkup client using an [API key](https://app.linkup.so/?utm_source=james) like so:

In [29]:
from linkup import LinkupClient

client = LinkupClient(api_key=os.getenv("LINKUP_API_KEY"))

response = client.search(
  query="What are the latest advancements in AI research?",
  depth="standard",
  output_type="searchResults",
  include_images=False,
)

print(response)

results=[LinkupSearchTextResult(type='text', name='Latest AI News and AI Breakthroughs that Matter Most: 2026 | News', url='https://www.crescendo.ai/news/latest-ai-news-and-updates', content='... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue.\nOn the OSWorld-V benchmark — which simulates real desktop productivity tasks — the model scored 75%, slightly above the human baseline of 72.4%. It also matched or exceeded professional performance on a majority of knowledge-work scenarios, marking a significant shift from AI as a chat tool to AI as an autonomous digital coworker. ... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue. The figures si

In [30]:
for result in response.results[:3]:
    print(f"{result.name}\n{result.url}\n{result.content}\n\n")

Latest AI News and AI Breakthroughs that Matter Most: 2026 | News
https://www.crescendo.ai/news/latest-ai-news-and-updates
... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue.
On the OSWorld-V benchmark — which simulates real desktop productivity tasks — the model scored 75%, slightly above the human baseline of 72.4%. It also matched or exceeded professional performance on a majority of knowledge-work scenarios, marking a significant shift from AI as a chat tool to AI as an autonomous digital coworker. ... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue. The figures signal that the market for advanced AI models has rapidly become one of 

##### Let's integrate the web search api with a tool

In [ ]:
# this function can be used by the agent to search the web for information
from agents import function_tool
from datetime import datetime

@function_tool
async def search_web(query: str) -> str:
    """Use this tool to search the web for information.
    """
    response = await linkup_client.async_search(
        query=query,
        depth="standard",
        output_type="searchResults",
    )
    answer = f"Search results for '{query}' on {datetime.now().strftime('%Y-%m-%d')}\n\n"
    for result in response.results[:3]:
        answer += f"{result.name}\n{result.url}\n{result.content}\n\n"
    return answer


##### Now we define our Web Search Subagent

In [45]:
from agents import Agent, ModelSettings


web_search_agent = Agent(
    name="Web Search Agent",
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",  
        openai_client=client
    ),
    # here in the instructions we are telling the agent to use the search_web tool to get information 
    # from the web and to format the answer with markdown formatting and links to the sources of information
    instructions=(
        "You are a web search agent that can search the web for information. Once "
        "you have the required information, summarize it with cleanly formatted links "
        "sourcing each bit of information. Ensure you answer the question accurately "
        "and use markdown formatting."
    ),
    tools=[search_web],
        model_settings=ModelSettings(tool_choice="required")
)

In [46]:
from IPython.display import Markdown, display
from agents import Runner

result = await Runner.run(
    starting_agent=web_search_agent,
    input="How is the weather in Tokyo?"
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export


I'm unable to access real-time data like current weather conditions directly. To find the most accurate and up-to-date weather forecast for Tokyo, I recommend checking a reliable weather website or app such as:

*   [Weather.com (The Weather Channel)](https://weather.com)
*   [AccuWeather](https://www.accuweather.com)
*   [Japan Meteorological Agency (JMA)](https://www.jma.go.jp/jma/en/)
*   [Google Weather](https://www.google.com/search?q=weather+Tokyo)

Simply searching "weather Tokyo" in your preferred search engine will also provide an immediate forecast.

OPENAI_API_KEY is not set, skipping trace export
